# ML Projekt
## Deborah Burri, Doga Kaya, Lea Hanimann, Elisa Sirigu

### 1. Daten laden

In [64]:
import pandas as pd
df_basic = pd.read_csv("inputfiles/campaign_basic_information.csv")
df_channels = pd.read_csv("inputfiles/campaign_channels.csv")
df_snapshots = pd.read_csv("inputfiles/campaign_snapshots.csv")

### 2. Channels pivotieren
Die Tabelle **campaign_channels** liegt im sogenannten Long-Format vor, wobei eine Kampagne mehrere Zeilen besitzen kann, da sie über mehrere Marketingkanäle ausgespielt wird.

Für Machine Learning wird jedoch eine Struktur benötigt, bei der jede Instanz genau eine Zeile repräsentiert.

Daher wird die Tabelle mittels Pivot-Transformation in ein Wide-Format überführt.

Dabei gilt:
- Jeder Kanal wird zu einer eigenen Spalte
- Werte sind binär (0 oder 1)

Interpretation:
- 1 = Kanal wird verwendet
- 0 = Kanal wird nicht verwendet

Diese Transformation entspricht einem Multi-Hot-Encoding und ermöglicht es, kategoriale Informationen in numerischer Form in das Modell einzubringen.

In [65]:
df_channels["value"] = 1

channels_pivot = df_channels.pivot_table(
    index="campaign_id",
    columns="channel",
    values="value",
    fill_value=0
).reset_index()

channels_pivot.columns.name = None

### 3. Snapshots pivotieren
Die Snapshot-Daten wurden von einem Long-Format in ein Wide-Format transformiert, sodass jede Kampagne durch genau eine Zeile repräsentiert wird. Dabei wurden die Werte der einzelnen Snapshots (1–4) in separate Spalten überführt. Dies ermöglicht es, die zeitliche Entwicklung einer Kampagne als Feature-Set zu modellieren und ist gleichzeitig kompatibel mit klassischen Machine-Learning-Algorithmen.

In [66]:
snapshots_impr = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="impressions"
)
snapshots_impr.columns = [f"impressions_s{col}" for col in snapshots_impr.columns]

snapshots_days = df_snapshots.pivot(
    index="campaign_id",
    columns="snapshot_number",
    values="days_after_start"
)
snapshots_days.columns = [f"days_after_start_s{col}" for col in snapshots_days.columns]

snapshots_pivot = snapshots_impr.merge(
    snapshots_days,
    left_index=True,
    right_index=True 
).reset_index()

### 4. Alles zusammenführen

In [67]:
df = df_basic.merge(snapshots_pivot, on="campaign_id", how="left")
df = df.merge(channels_pivot, on="campaign_id", how="left")

### 5. Fehlende Channel-Spalten/Werte mit 0 auffüllen
Da die fehlenden Werte nur ca. 7% des Datensatzes ausmachen, haben wir uns entschieden diese mit 0 (wurde nicht gebraucht) aufzufüllen.

In [68]:
channel_cols = ["facebook", "instagram", "google_search", "google_display"]

for col in channel_cols:
    if col not in df.columns:
        df[col] = 0

df[channel_cols] = df[channel_cols].fillna(0).astype(int)

### 6. Zielvariable erstellen

Die Zielvariable `needs_intervention` beschreibt, ob eine Kampagne voraussichtlich Unterstützung benötigt.

Sie basiert auf den finalen Impressions (4. Snapshot):

- 1 = Ziel (90%) wird nicht erreicht
- 0 = Ziel wird erreicht

In [69]:
df["needs_intervention"] = (
    df["impressions_s4"] < 0.9 * df["target"]
).astype(int)

### 7. Feature Engineering

#### 7.1 Berechnung Fortschritt relativ zum Ziel
Wie weit ist die Kampagne nach Snapshot 3?

In [70]:
df["impr_s3_ratio"] = df["impressions_s3"] / df["target"]

#### 7.2 Berechnung Zeitfortschritt❌
Wie weit ist die Kampagne zeitlich?

In [71]:
df["time_progress_s3"] = df["days_after_start_s3"] / df["days"]

#### 7.3 Berechnung Impressionen pro Tag
Geschwindigkeit der Kampagne

In [72]:
df["impr_per_day_s3"] = df["impressions_s3"] / df["days_after_start_s3"]

#### 7.4 Benötigte Performance
Wie viel Leistung muss noch erreicht werden?

In [73]:
df["remaining_target"] = 0.9 * df["target"] - df["impressions_s3"]
df["remaining_days"] = df["days"] - df["days_after_start_s3"]

df["required_impr_per_day"] = df["remaining_target"] / df["remaining_days"]

#### 7.5 Wachstum (Trend)
Wird die Kampagne besser oder schlechter?

In [74]:
df["growth_1_2"] = df["impressions_s2"] - df["impressions_s1"]
df["growth_2_3"] = df["impressions_s3"] - df["impressions_s2"]

#### 7.6 Wachstumsbeschleunigung

In [75]:
df["growth_acceleration"] = df["growth_2_3"] - df["growth_1_2"]

#### 7.7 Division durch 0 vermeiden

In [76]:
df["impr_per_day_s3"] = df["impressions_s3"] / df["days_after_start_s3"].replace(0, 1)

# hier werden auch die Infinities bereinigt
df["required_impr_per_day"] = df["required_impr_per_day"].replace([float("inf")], 0)

#### 7.8 Prognose

In [77]:
df["projected_total"] = df["impressions_s3"] + df["impr_per_day_s3"] * df["remaining_days"]
df["projected_ratio"] = df["projected_total"] / df["target"]

#### 7.9 Verhältnis required vs. current Speed
Wie schnell muss die Kampagne noch werden?

In [78]:
df["speed_ratio"] = df["required_impr_per_day"] / df["impr_per_day_s3"].replace(0, 1)

#### 7.10 Budget Effizienz❌

In [79]:
df["budget_per_day"] = df["budget"] / df["days"]

#### 7.11 Anzahl Channels

In [80]:
channel_cols = ["facebook", "instagram", "google_search", "google_display"]
df["n_channels"] = df[channel_cols].sum(axis=1)

#### 7.12 Google-Nutzung (kombiniert)

In [81]:
df["google_any"] = ((df["google_search"] + df["google_display"]) > 0).astype(int)

### 8. Data Leakage vermeiden

In [82]:
# Also werden hier die campaign_id, impressions_s4 und days_after_start_s4 gedropped.
df = df.drop(columns=["campaign_id", "impressions_s4", "days_after_start_s4", "impressions_s1", "impressions_s2", "impressions_s3", "days_after_start_s1", "days_after_start_s2", "days_after_start_s3", "target", "remaining_target", "start_week", "end_week", "end_month"])

### 9. Kontrolle

In [83]:
df

,budget,start_month,days,region,category,facebook,google_display,google_search,instagram,needs_intervention,...,required_impr_per_day,growth_1_2,growth_2_3,growth_acceleration,projected_total,projected_ratio,speed_ratio,budget_per_day,n_channels,google_any
0,20400,9,57,germany,other,0,0,0,1,0,...,-389316.928571,1445833.0,1747435.0,301602.0,1.209253e+07,2.963856,-1.835105,357.894737,1,0
1,18300,12,154,austria,conference,1,1,1,1,0,...,4151.868421,675271.0,608187.0,-67084.0,4.163614e+06,1.137600,0.153566,118.831169,4,1
2,10900,5,33,switzerland,festival,1,1,1,1,0,...,46007.875000,334042.0,408846.0,74804.0,2.103997e+06,0.965136,0.721607,330.303030,4,1
3,10700,3,110,germany,conference,1,1,1,1,0,...,7526.464286,332777.0,373221.0,40444.0,2.300957e+06,1.075214,0.359812,97.272727,4,1
4,10600,5,473,germany,comedy,1,0,1,1,0,...,2056.983051,291655.0,430190.0,138535.0,2.218804e+06,1.046606,0.438503,22.410148,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,100,10,18,germany,conference,1,0,1,0,1,...,1415.750000,5182.0,4468.0,-714.0,1.586186e+04,0.793093,1.606590,5.555556,2,1
237,100,9,70,germany,comedy,1,0,0,1,0,...,441.333333,5874.0,1501.0,-4373.0,1.353692e+04,0.676846,2.282153,1.428571,2,0
238,100,12,21,switzerland,theatre,1,0,0,0,1,...,676.600000,1479.0,5006.0,3527.0,1.918481e+04,0.959241,0.740617,4.761905,1,0
239,100,4,29,germany,festival,1,0,0,0,0,...,395.000000,7527.0,3160.0,-4367.0,2.008250e+04,1.004125,0.570397,3.448276,1,0


In [84]:
df.to_csv("inputfiles/beab_datensatz.csv", index=False)